# 🛡️ Fine-Tuning Lightweight Prompt Injection Classifier (DeBERTa-v3-small)
**Project:** Prompt-Injection Benchmark & Lightweight Defense for Tool-Using LLM Agents  
**Author:** Saksham  
**Dataset:** 11,683 Real-World Prompt Injection & Benign Samples (Hugging Face / Kaggle benchmarks)  
**Hardware:** Google Colab Free T4 GPU

### Step 1: Ensure T4 GPU is Enabled
In the top menu, click **Runtime** ➔ **Change runtime type** ➔ select **T4 GPU** ➔ **Save**.

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets accelerate scikit-learn evaluate pandas pyarrow
import torch
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))

### Step 2: Acquire Real Dataset (11,683 Real Samples)
This automatically downloads the official real-world prompt injection datasets from Hugging Face Hub (deepset + S-Labs).

In [ ]:
import urllib.request
import pandas as pd
import os

print('Downloading official open-source prompt injection datasets...')
# 1. Download deepset/prompt-injections
url_deepset_train = 'https://huggingface.co/datasets/deepset/prompt-injections/resolve/main/data/train-00000-of-00001-9564e8b05b4757ab.parquet'
url_deepset_test = 'https://huggingface.co/datasets/deepset/prompt-injections/resolve/main/data/test-00000-of-00001-701d16158af87368.parquet'
# 2. Download S-Labs dataset
url_slabs = 'https://huggingface.co/datasets/S-Labs/prompt-injection-dataset/resolve/main/data/train.csv'

urllib.request.urlretrieve(url_deepset_train, 'deepset_train.parquet')
urllib.request.urlretrieve(url_deepset_test, 'deepset_test.parquet')
urllib.request.urlretrieve(url_slabs, 'slabs_train.csv')

df_deepset = pd.concat([pd.read_parquet('deepset_train.parquet'), pd.read_parquet('deepset_test.parquet')], ignore_index=True)
df_slabs = pd.read_csv('slabs_train.csv')

combined = pd.concat([df_deepset, df_slabs], ignore_index=True)
combined['text'] = combined['text'].astype(str).str.strip()
combined = combined.drop_duplicates(subset=['text'])
combined = combined[combined['text'].str.len() > 10]

combined.to_csv('real_classifier_training_data.csv', index=False)
print(f'Dataset ready: {len(combined)} real samples loaded!')
print(combined['label'].value_counts())

### Step 3: Configure Tokenizer & DeBERTa-v3-small Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

MODEL_NAME = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

df = pd.read_csv('real_classifier_training_data.csv')
raw_ds = Dataset.from_dict({'text': df['text'].tolist(), 'label': df['label'].tolist()})

print('Tokenizing dataset...')
tokenized_ds = raw_ds.map(
    lambda b: tokenizer(b['text'], truncation=True, max_length=512, padding='max_length'),
    batched=True,
    batch_size=1000
)
split_ds = tokenized_ds.train_test_split(test_size=0.15, seed=42)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    probs = exp_logits[:, 1] / np.sum(exp_logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    try:
        auroc = roc_auc_score(labels, probs)
    except Exception:
        auroc = 0.5
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall, 'auroc': auroc}

print('Tokenization complete. Ready for training!')

### Step 4: Train on Free T4 GPU (Takes ~10–12 minutes)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    fp16=False,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_ds['train'],
    eval_dataset=split_ds['test'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print('Starting fine-tuning...')
train_result = trainer.train()
print('Training complete! Final evaluation metrics:')
eval_metrics = trainer.evaluate()
for k, v in eval_metrics.items():
    print(f'  {k}: {v:.4f}')

### Step 5: Package & Download Model Weights
This packages your fine-tuned model into `injection_classifier.zip` and triggers an automatic browser download.

In [ ]:
from google.colab import files

SAVE_DIR = './injection_classifier'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -r injection_classifier.zip ./injection_classifier
print('Model zipped successfully! Downloading now...')
files.download('injection_classifier.zip')